In [1]:
!apt-get -y install ffmpeg
!pip install -U gradio transformers soundfile librosa


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 35.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: transformers
    Found existing installation: transformer

In [2]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
!pip -q install -U transformers gradio librosa soundfile accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.6 MB/s eta 0:00:00


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch


In [ ]:
!pip -q install -U transformers gradio librosa soundfile accelerate bitsandbytes

import torch
import numpy as np
import librosa
import gradio as gr

from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor,
    AutoTokenizer, AutoModelForCausalLM
)

# =========================
# PATHS (Drive / Local)
# =========================
WHISPER_MODEL_PATH  = "/content/drive/MyDrive/Bitirme_projesi/son_kodlar/son_kodlar_tokeribrahim/whisper_model"
WAV2VEC2_MODEL_PATH  = "/content/drive/MyDrive/Bitirme_projesi/son_kodlar/wav2vec2_son"

# LLaMA: LoRA merge edilmişse direkt bu klasör; LoRA ayrıysa aşağıda not var.
LLAMA_MODEL_PATH    = "/content/drive/MyDrive/Bitirme_projesi/son_kodlar/llama_3/llama_model/lora_adapter"
BASE_ID = "meta-llama/Meta-Llama-3-8B-Instruct"   # base
ADAPTER_DIR = LLAMA_MODEL_PATH                   # adapter klasörü
# =========================
# LABELS (6 class)
# =========================
TARGET_LABELS = ["anger", "disgust", "fear", "happy", "sad", "surprise"]
id2label = {i: l for i, l in enumerate(TARGET_LABELS)}
label2id = {l: i for i, l in id2label.items()}

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)
use_fp16 = (device.type == "cuda")

# =========================
# 1) WHISPER LOAD
# =========================
whisper_processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_PATH)
whisper_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_PATH).to(device)
if use_fp16:
    whisper_model = whisper_model.half()
whisper_model.eval()

print("WHISPER ->", next(whisper_model.parameters()).device, next(whisper_model.parameters()).dtype)

# =========================
# 2) WAV2VEC2 LOAD
# =========================
wav2vec2_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAV2VEC2_MODEL_PATH)
wav2vec2_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    WAV2VEC2_MODEL_PATH,
    num_labels=len(TARGET_LABELS),
    id2label=id2label,
    label2id=label2id
).to(device)
wav2vec2_model.eval()

print("WAV2VEC2 ->", next(wav2vec2_model.parameters()).device, next(wav2vec2_model.parameters()).dtype)

# =========================
# 3) LLaMA LOCAL LOAD (4-bit)
# =========================
bnb = BitsAndBytesConfig(load_in_4bit=True)

llama_tokenizer = AutoTokenizer.from_pretrained(BASE_ID, use_fast=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE_ID, device_map="auto", quantization_config=bnb, dtype=torch.float16
)
llama_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
llama_model.eval()

print("LLAMA loaded (4bit)")

# =========================
# HELPERS
# =========================
def load_audio_16k_mono(audio_path: str):
    wav, sr = librosa.load(audio_path, sr=16000, mono=True)
    return wav.astype(np.float32), 16000

def whisper_transcribe(audio_path):
    wav, sr = load_audio_16k_mono(audio_path)
    inputs = whisper_processor(wav, sampling_rate=sr, return_tensors="pt")
    x = inputs.input_features.to(device)
    if use_fp16:
        x = x.half()
    pred_ids = whisper_model.generate(x, max_new_tokens=128)
    return whisper_processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()

def wav2vec2_emotion(audio_path):
    wav, sr = load_audio_16k_mono(audio_path)
    fe = wav2vec2_extractor(wav, sampling_rate=sr, return_tensors="pt", padding=True)
    input_values = fe["input_values"].to(device)
    attention_mask = fe.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    with torch.no_grad():
        logits = wav2vec2_model(input_values=input_values, attention_mask=attention_mask).logits

    probs = torch.softmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()
    pred_id = int(np.argmax(probs))
    return id2label[pred_id], {id2label[i]: float(probs[i]) for i in range(len(TARGET_LABELS))}

def llama_prompt(text):
    # Basit ve sağlam prompt: sadece label döndürmeye zorlar
    labels = ", ".join(TARGET_LABELS)
    return (
        "You are an emotion classification assistant.\n"
        f"Classify the emotion of the following text into exactly one of these labels:\n"
        f"{labels}\n\n"
        f"Text: {text}\n\n"
        "Answer with only the label:"
    )

def llama_text_emotion(text):
    prompt = llama_prompt(text)

    inputs = llama_tokenizer(prompt, return_tensors="pt").to(llama_model.device)

    with torch.no_grad():
        out = llama_model.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            temperature=0.0,
            eos_token_id=llama_tokenizer.eos_token_id,
            pad_token_id=llama_tokenizer.pad_token_id
        )

    decoded = llama_tokenizer.decode(out[0], skip_special_tokens=True)
    # prompt + cevap birlikte gelir; sadece son kısmı al
    answer = decoded.split("Answer with only the label:")[-1].strip().lower()

    # güvenli normalize (bazı modeller nokta/boşluk koyar)
    answer = answer.replace(".", "").replace(",", "").strip()

    # label dışı bir şey dönerse en yakın eşleşme
    if answer not in TARGET_LABELS:
        # basit fallback: ilk geçen label’ı yakala
        for lab in TARGET_LABELS:
            if lab in answer:
                return lab
        return "unknown"

    return answer

def run_all(audio_path):
    if audio_path is None:
        return "Ses alınamadı.", "", {}, "", "—"

    text = whisper_transcribe(audio_path)

    audio_label, audio_probs = wav2vec2_emotion(audio_path)

    text_label = llama_text_emotion(text) if text else "unknown"

    agreement = "✅ SAME" if (text_label == audio_label and text_label != "unknown") else "❌ DIFFERENT"
    return text, audio_label, audio_probs, text_label, agreement

# =========================
# UI
# =========================
with gr.Blocks() as demo:
    gr.Markdown("## 🎙️ Whisper (TR) + 🎭 wav2vec2 (Audio Emotion) + 🧠 LLaMA (Text Emotion) — Local")
    audio = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Ses yükle veya kaydet")

    btn = gr.Button("Run (Transcribe + Emotions)")

    out_text = gr.Textbox(label="Whisper Transkript", lines=4)
    out_audio_emotion = gr.Textbox(label="wav2vec2 Audio Emotion", lines=1)
    out_audio_probs = gr.Label(label="wav2vec2 Probabilities (6 sınıf)")
    out_text_emotion = gr.Textbox(label="LLaMA Text Emotion", lines=1)
    out_agree = gr.Textbox(label="Agreement", lines=1)

    btn.click(run_all, inputs=audio, outputs=[out_text, out_audio_emotion, out_audio_probs, out_text_emotion, out_agree])

demo.launch(share=True, debug=True)


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

WHISPER -> cuda:0 torch.float16


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

WAV2VEC2 -> cuda:0 torch.float32


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

LLAMA loaded (4bit)
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c21fb9ccd07f6c4db3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'trans